In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import trim, col

catalog_name = "my_catalog"
#Load Data Dictionary
data_dictionary_path = (
    "/Volumes/my_catalog/landing/raw_files/data_dictionary.csv"
)

dictionary_df = (
    spark.read
         .option("header", True)
         .csv(data_dictionary_path)
         .select(
             trim(col("schema_name")).alias("schema_name"),
             trim(col("table_name")).alias("table_name"),
             trim(col("business_description")).alias("description")
         )
)

#Update Table Descriptions
results = []

for row in dictionary_df.collect():

    schema_name = row["schema_name"]
    table_name = row["table_name"]
    description = row["description"]

    full_table_name = (
        f"{catalog_name}.{schema_name}.{table_name}"
    )

    try:

        # Verify table exists
        spark.sql(f"""
        DESCRIBE TABLE {full_table_name}
        """)

        # Escape single quotes
        description_sql = description.replace("'", "''")

        # Update comment
        spark.sql(f"""
        COMMENT ON TABLE {full_table_name}
        IS '{description_sql}'
        """)

        print(f"Updated: {full_table_name}")

        results.append(
            Row(
                table=full_table_name,
                status="SUCCESS",
                description=description
            )
        )

    except Exception as e:

        print(f"Failed: {full_table_name}")

        results.append(
            Row(
                table=full_table_name,
                status="FAILED",
                description=description,
                error=str(e)
            )
        )